In [3]:

# ANALISIS DE MÉTRICAS DE DIVERSIDAD
# Este script aplica un conjunto de enfoques complementarios para caracterizar
# la diversidad alfa en tu repertorio: primero calcula los números de Hill 
# mediante alphaDiversity de alakazam ; luego incorpora métricas adicionales 
# con vegan como índices de diversidad y equidad para describir la distribución
# de abundancias; y finalmente utiliza ineq para estimar desigualdad clonal a través
# del índice de Gini y otras medidas de concentración. Al combinar estas diez métricas, 
# obtienes una visión integrada de la cantidad, equilibrio y desigualdad en la arquitectura 
# clonal de tu repertorio.

In [4]:
# Paquetes y librerías
library(readr)
library(dplyr)
library(ggplot2)
library(viridisLite)
library(here)  # para rutas relativas
# install.packages("alakazam")
library(alakazam)
library(ineq)
library(vegan)
library(tidyr)
library(tibble)

here() starts at /Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/notebooks

Warning message:
"replacing previous import 'S4Arrays::makeNindexFromArrayViewport' by 'DelayedArray::makeNindexFromArrayViewport' when loading 'SummarizedExperiment'"


To cite the alakazam package in publications, please use:

  Gupta N, Vander Heiden J, Uduman M, Gadala-Maria D, Yaari G,
  Kleinstein S (2015). "Change-O: a toolkit for analyzing large-scale B
  cell immunoglobulin repertoire sequencing data." _Bioinformatics_,
  1-3. doi:10.1093/bioinformatics/btv359
  <https://doi.org/10.1093/bioinformatics/btv359>.

To cite the Ig-specific lineage reconstruction and diversity methods,
please use:

  Stern J, Yaari G, Vander Heiden J, Church G, Donahue W, Hintzen R,
  Huttner A, Laman J, Nagra R, Nylander A, Pitt D, Ramanan S, Siddiqui
  B, Vigneault F, Kleinstein S, Hafler D, O'Connor K (2014). "B cells
  populating the multiple sclerosis brain mature in the draining
  cervical lymph nodes." _Science 

In [5]:

# Cargar tu archivo .tsv
archivo_clones <- "../data/output/repertorio_E_insilico_1000_seqs_clone-pass.tsv"
clones <- read_tsv(archivo_clones)
# Agregar identificador de muestra (porque es simulado)
clones <- clones %>%
  mutate(sample_id = "repertorio_simulado")


# Contar secuencias por clon y muestra
clone_counts <- clones %>%
  group_by(sample_id, clone_id) %>%
  summarise(count = n(), .groups = "drop")

Rows: 1000 Columns: 50
-- Column specification --------------------------------------------------------
Delimiter: "\t"
chr (20): sequence_id, sequence, v_call, d_call, j_call, sequence_alignment,...
dbl (25): junction_length, np1_length, np2_length, v_sequence_start, v_seque...
lgl  (5): rev_comp, productive, stop_codon, vj_in_frame, c_call

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [6]:
hill <- alphaDiversity(
  data = clone_counts,
  clone = "clone_id",
  min_q = 0,
  max_q = 4,
  step_q = 1,
  nboot = 100,
  ci = 0.95
)

# Extraer la tabla
df <- hill@diversity

# Agregar columna con índices clásicos
df <- df %>%
  dplyr::mutate(
    indice_clasico = case_when(
      q == 0 ~ d,            # riqueza observada
      q == 1 ~ log(d),       # Shannon clásico H = ln(D1)
      q == 2 ~ 1/d,          # Simpson clásico D = 1/D2
      q == 3 ~ 1/(d^2),      # ∑ p_i^3 = 1/(D3^2)
      q == 4 ~ 1/(d^3)       # ∑ p_i^4 = 1/(D4^3)
    )
  )

print(df)
df_tbl <- as_tibble(df)

# A tibble: 5 x 10
# Groups:   group [1]
  group     q     d  d_sd d_lower d_upper     e e_lower e_upper indice_clasico
  <chr> <dbl> <dbl> <dbl>   <dbl>   <dbl> <dbl>   <dbl>   <dbl>          <dbl>
1 All       0  994. 0.943    992.    996. 1       0.998    1.00        9.94e+2
2 All       1  994. 1.30     991.    996. 1.000   0.997    1.00        6.90e+0
3 All       2  993. 1.87     989.    997. 0.999   0.995    1.00        1.01e-3
4 All       3  992. 2.79     987.    998. 0.998   0.993    1.00        1.02e-6
5 All       4  991. 4.28     982.    999. 0.996   0.988    1.00        1.03e-9


In [7]:
hill_numbers <- function(clones_df){
    clone_counts <- clones_df %>%
        group_by(sample_id, clone_id) %>%
        summarise(count = n(), .groups = "drop")
    hill <- alphaDiversity(
        data = clone_counts,
        clone = "clone_id",
        min_q = 0,
        max_q = 4,
        step_q = 1,
        nboot = 100,
        ci = 0.95
    )
    return(hill@diversity)
}
rep_hill_numbers <- hill_numbers(clones)
rep_hill_numbers

group,q,d,d_sd,d_lower,d_upper,e,e_lower,e_upper
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
All,0,994.1500,0.9252354,992.3366,995.9634,1.0000000,0.9981759,1.001824
All,1,993.8232,1.2802723,991.3139,996.3325,0.9996713,0.9971472,1.002195
All,2,993.3063,1.8408079,989.6984,996.9142,0.9991513,0.9955222,1.002780
All,3,992.4711,2.7433908,987.0941,997.8480,0.9983112,0.9929026,1.003720
All,4,991.1004,4.2153249,982.8385,999.3623,0.9969325,0.9886220,1.005243


In [8]:
richness <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 0) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

richness(rep_hill_numbers)

[1] 994.15

In [9]:
 q1 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q1(rep_hill_numbers)

[1] 993.8232

In [10]:
shannon <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(log(metric_value))
} 

shannon(rep_hill_numbers)

[1] 6.901559

In [11]:
 q2 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q2(rep_hill_numbers)

[1] 993.3063

In [12]:
simpson <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value))
} 

simpson(rep_hill_numbers)

[1] 0.001006739

In [13]:
 q3 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 3) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q3(rep_hill_numbers)

[1] 992.4711

In [14]:
d3 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value)^2)
} 

d3(rep_hill_numbers)

[1] 1.012469e-06

In [15]:
 q4 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 4) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

q4(rep_hill_numbers)

[1] 991.1004

In [16]:
d4 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 4) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value)^3)
} 

d4(rep_hill_numbers)

[1] 1.027181e-09

In [17]:
# MÉTRICA CHAO1 y ACE PAQUETE VEGAN
metricas_chao1ace <- clone_counts %>%
  group_by(sample_id) %>% 
  summarise(
    chao1 = estimateR(count)["S.chao1"],
    ace   = estimateR(count)["S.ACE"]
  )

print(metricas_chao1ace)

# A tibble: 1 x 3
  sample_id            chao1    ace
  <chr>                <dbl>  <dbl>
1 repertorio_simulado 82588. 99500.


In [18]:


chao1 <- function(clones_df){
  clone_counts <- clones_df %>%
    group_by(sample_id, clone_id) %>%
    summarise(count = n(), .groups = "drop")
  
  metricas_chao1 <- clone_counts %>%
    group_by(sample_id) %>%
    summarise(
      chao1 = as.numeric(vegan::estimateR(count)["S.chao1"]),
      .groups = "drop"
    )%>%
    dplyr::pull(chao1)
  
  return(metricas_chao1[1])
}

# Ejecutar
rep_chao1 <- chao1(clones)
print(rep_chao1)


[1] 82587.5


In [19]:
ace <- function(clones_df){
  clone_counts <- clones_df %>%
    group_by(sample_id, clone_id) %>%
    summarise(count = n(), .groups = "drop")
  
  metricas_ace <- clone_counts %>%
    group_by(sample_id) %>%
    summarise(
      ace = as.numeric(vegan::estimateR(count)["S.ACE"]),
      .groups = "drop"
    )%>%
    dplyr::pull(ace)
  
  return(metricas_ace[1])
}

# Ejecutar
rep_ace <- ace(clones)
print(rep_ace)

[1] 99500


In [20]:
# MÉTRICA GINI PAQUETE INEQ
calc_gini <- function(df) {
  ineq::ineq(df$count, type = "Gini")
}
gini_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(
    gini = calc_gini(cur_data())
  )

print(gini_result)

Warning message:
"There was 1 warning in `summarise()`.
i In argument: `gini = calc_gini(cur_data())`.
i In group 1: `sample_id = "repertorio_simulado"`.
Caused by warning:
! `cur_data()` was deprecated in dplyr 1.1.0.
i Please use `pick()` instead."


# A tibble: 1 x 2
  sample_id              gini
  <chr>                 <dbl>
1 repertorio_simulado 0.00497


In [21]:
gini <- function (clones_df){
    clone_counts <- clones_df %>%
        group_by(sample_id, clone_id) %>%
        summarise(count = n(), .groups = "drop")
    metrica_gini <- ineq::ineq(clone_counts$count, type = "Gini")
    return(metrica_gini)
}
gini(clones)

[1] 0.004974874

In [22]:
# MÉTRICA PIELOU PAQUETE VEGAN

calc_pielou <- function(df) {
  abund <- df$count
  H <- diversity(abund, index = "shannon")  # Shannon
  S <- specnumber(abund)                    # número de clones
  J <- H / log(S)                           # Pielou
  return(J)
}

pielou_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(
    pielou = calc_pielou(cur_data())
  )

print(pielou_result)


# A tibble: 1 x 2
  sample_id           pielou
  <chr>                <dbl>
1 repertorio_simulado  1.000


In [23]:
pielou <- function(clones_df){
 
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  H <- vegan::diversity(clone_counts$count, index = "shannon")
  S <- vegan::specnumber(clone_counts$count)
  J <- H / log(S)
  
  return(as.numeric(J))  # 👈 devuelve solo el número
}

pielou(clones)

[1] 0.999722

In [24]:
# MÉTRICA BASHARIN FUNCIONES R+ VEGAN

calc_basharin <- function(df) {
  abund <- df$count
  N <- sum(abund)
  S <- specnumber(abund)
  
  # Casos triviales
  if (N == 0 || S <= 1) return(0)
  
  H <- diversity(abund, index = "shannon")
  print(H)
  basharin <- H + (S - 1) / (2 * N)
  return(basharin)
}

# Aplicar por muestra
basharin_result <- clone_counts %>%
  group_by(sample_id) %>%
  summarise(basharin = calc_basharin(cur_data()), .groups = "drop")

print(basharin_result)


[1] 6.900824
# A tibble: 1 x 2
  sample_id           basharin
  <chr>                  <dbl>
1 repertorio_simulado     7.40


In [25]:
basharin <- function(clones_df){
  
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  abund <- clone_counts$count
  N <- sum(abund)
  S <- vegan::specnumber(abund)
  
  if (N == 0 || S <= 1) return(0)
  
  # Shannon
  H <- vegan::diversity(abund, index = "shannon")
  
  # Basharin
  basharin_val <- H + (S - 1) / (2 * N)
  
  return(as.numeric(basharin_val))  # 👈 devuelve número puro
}
basharin(clones)

[1] 7.397824

In [26]:
d50_fun <- function(counts) {
  counts <- sort(counts, decreasing = TRUE)
  total <- sum(counts)
  cum <- cumsum(counts)
  which(cum >= 0.5 * total)[1]
}

# Calcular D50
d50_val <- d50_fun(clone_counts$count)
print(d50_val)
d50_result <- tibble::tibble(D50 = d50_val)


[1] 495


In [27]:
d50 <- function(clones_df){
 
  clone_counts <- clones_df %>%
    dplyr::group_by(sample_id, clone_id) %>%
    dplyr::summarise(count = n(), .groups = "drop")
  
  counts <- sort(clone_counts$count, decreasing = TRUE)
  total  <- sum(counts)
  cum    <- cumsum(counts)
  
  d50_val <- which(cum >= 0.5 * total)[1]
  
  return(as.numeric(d50_val))  # 👈 devuelve número puro
}
d50(clones)

[1] 495

In [28]:
metricas_diversidad <- c(richness= richness(rep_hill_numbers),q1= q1(rep_hill_numbers),shannon= shannon(rep_hill_numbers), q2= q2(rep_hill_numbers), simpson= simpson(rep_hill_numbers), q3= q3(rep_hill_numbers), 
d3= d3(rep_hill_numbers), q4= q4(rep_hill_numbers), d4= d4(rep_hill_numbers), chao1= chao1(clones), ace= ace(clones), gini= gini(clones), pielou= pielou(clones), basharin= basharin(clones), d50= d50(clones))
metricas_diversidad


richness           q1      shannon           q2      simpson           q3 
9.941500e+02 9.938232e+02 6.901559e+00 9.933063e+02 1.006739e-03 9.924711e+02 
          d3           q4           d4        chao1          ace         gini 
1.012469e-06 9.911004e+02 1.027181e-09 8.258750e+04 9.950000e+04 4.974874e-03 
      pielou     basharin          d50 
9.997220e-01 7.397824e+00 4.950000e+02

In [31]:
tabla_diversidad <- bind_rows(metricas_diversidad)

tabla_diversidad$sample_id <- "E_1000seq"
tabla_diversidad$escenario <- "E"
tabla_diversidad$size <- 1000

In [32]:
readr::write_tsv(tabla_diversidad, "../results/diversity_metrics/diversity_E_1000seqs.tsv")